# Open Notebook in Colab 

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/childmindresearch/llm_tracker/blob/main/tutorials/discovery_tutorial.ipynb)


Please cite this if you use llm_tracker: <br>
Low, D.M., Mair, P., Nock, M., & Ghosh, S. (2026). Text Psychometrics: Assessing Psychological Constructs in Text Using Natural Language Processing. PLOS Digital Health.

# Discover constructs and build a codebook

Use this tutorial to explore psychological constructs in your texts before you
have a codebook. You choose a theoretical framework, such as cognitive behavioral
therapy. A large language model (LLM) then suggests constructs supported by the
texts and organizes related ideas into groups. You review these suggestions,
make changes, and create a codebook for coding other documents.

Work through the notebook from top to bottom. Each code cell is a block of
instructions with a Run button. The setup cell below prepares the software.
If you update the software while the notebook is open, restart the Colab session
or Jupyter kernel (the process that runs the code), then start again from setup.

Finding constructs and grouping related ideas send requests to the LLM service
and may use paid credits. Reviewing groups, setting aside constructs, preparing
tables, and creating the codebook do not send additional requests. An optional
later step asks the LLM to review group names and definitions. The notebook does
not save your API key, the access key for the LLM service.


# Setup (first time only)

Run the next cell to download LLM Tracker and the practice files, and install
the software needed for this tutorial. You do not need to download the project
yourself beforehand. In Google Colab, you can start with this notebook and an
internet connection.

You can also use Jupyter on your own computer with Python 3.11–3.13 installed.
Setup may take a few minutes. If asked to restart the Colab session or Jupyter
kernel, do so and run this setup cell again before continuing. Setup does not
send requests to an LLM or spend API credits.

In [1]:
from pathlib import Path
import subprocess
import sys
import tempfile
from urllib.request import urlretrieve
from zipfile import ZipFile
import os 
from getpass import getpass

# Find an existing copy, or download LLM Tracker and the sample data.
project_dir = Path.cwd()
if not (project_dir / "src" / "llm_tracker").is_dir():
    project_dir = project_dir.parent

if not (project_dir / "src" / "llm_tracker").is_dir():
    project_dir = Path.cwd() / "llm_tracker-main"
    if not (project_dir / "src" / "llm_tracker").is_dir():
        print("Downloading LLM Tracker...")
        with tempfile.TemporaryDirectory() as folder:
            archive = Path(folder) / "llm_tracker.zip"
            urlretrieve(
                "https://codeload.github.com/childmindresearch/llm_tracker/zip/refs/heads/main",
                archive,
            )
            with ZipFile(archive) as downloaded:
                downloaded.extractall(project_dir.parent)

# Allow NumPy 2 for Python 3.13 compatibility.
pyproject = project_dir / "pyproject.toml"
settings = pyproject.read_text()
settings = settings.replace('numpy = "^1.26"', 'numpy = ">=1.26,<3"')
pyproject.write_text(settings)

# Install the package; stop if installation fails.
subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--only-binary=numpy,pandas,scikit-learn",
        "--timeout", "30", "--retries", "1",
        "-e", str(project_dir), "openpyxl", "xlrd", "jinja2",
    ],
    check=True,
)

sys.path.insert(0, str(project_dir / "src"))

# Make the bundled sample-data paths work.
%cd "{project_dir}"

import llm_tracker.discovery
print("Discovery setup complete.")


Obtaining file:///Users/freymon.perez/Projects/GitHub/llm_tracker
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for llm-tracker (pyproject.toml): started
  Building editable for llm-tracker (pyproject.toml): finished with status 'done'
  Created wheel for llm-tracker: filename=llm_tracker-0.1.0b1-py3-none-any.whl size=10615 sha256=ca20f5fa06aadcd34b6e1d646c5092e2f4e07a421fc3faf1aec30da96b843a1f
  Stored in directory: /private/var/folders/09/drdv5jw504142vpxfq711jf815q1qc/T/pip-ephem-wheel-cache-m505yvs7/wheels/9d/32/98/c6659


[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
/Users/freymon.perez/Projects/GitHub/llm_tracker/src/llm_tracker/models.py:40: UserWarning: Field name "construct" in "ConstructInstance" shadows an attribute in parent "BaseModel"
  class ConstructInstance(BaseModel):


Discovery setup complete.


## 1. Choose your data and settings

Choose how to provide your texts by changing `input_source` in the next cell.
A path is the address of a file or folder.

| Choose | What happens |
|---|---|
| `"demo"` | Creates six short practice texts. No file upload or Google Drive connection is needed. |
| `"threshold_demo"` | Uses the small synthetic dataset with repeated examples for the document-count and prototypicality filters. The LLM still determines the final scores. |
| `"path"` | Opens a file or folder at the address in `input_path`. The path defaults to the 18-document synthetic dataset; for your own file in the project's `data/` folder, use `input_path = Path("data/my_data.csv")`. If running Jupyter on your computer, you can also give the full address of a file there. |
| `"upload"` | **Colab only:** lets you choose one file from your computer. Colab keeps a temporary copy; it disappears when the Colab session's storage is cleared. |
| `"drive"` | **Colab only:** asks permission to connect to Google Drive, then opens the file or folder at `drive_path`. For a file in a Research folder in My Drive, use an address such as `/content/drive/MyDrive/Research/my_data.csv`. |

For your own data, set `text_column` to the heading of the column containing
your texts. If you have a column of document identifiers, enter its heading as
`id_column`; otherwise leave `id_column=None`. The demo fills these in for you.

An address such as `data/my_data.csv` starts from the project folder prepared
during setup. In Colab, files on your computer are not available automatically.
Use Colab's Files sidebar to place a file in `data/`, choose `"upload"`, or
choose `"drive"` to read it from Google Drive.

- **Text files (TXT):** each file is one document. Set `text_column=None` and
  `id_column=None`.
- **Spreadsheets (CSV, XLSX, or XLS):** set `text_column` so that each row is
  treated as a separate document. Without it, the entire spreadsheet is treated
  as one document.
- **Folders:** files directly inside the folder are read, but files inside
  further subfolders are not. All spreadsheets must use the same column settings.

Results are saved separately from your original data. `output_root` is the
main results folder. When using `"drive"`, you can set it to an address inside
`/content/drive/MyDrive/` to keep results in Google Drive. Otherwise, download
your results before Colab clears the session's storage.

`model_name` selects the LLM. The default, `google/gemini-flash-latest`, uses
Gemini Flash latest through OpenRouter, the service connecting this notebook
to the model. You can enter another model's OpenRouter name instead.
The key-entry cell uses `OPENROUTER_API_KEY` if you have already made your
access key available under that name. Otherwise, it asks you to enter the key
without showing its characters on screen.


In [2]:
from pathlib import Path

input_source = "threshold_demo"  # Choose "demo", "threshold_demo", "path", "upload", or "drive".

framework = "Modern clinical psychology constructs (CBT, DBT, ACT, etc.)"  # Other options: "Cognitive behavioral therapy" "Political beliefs" "Buddhism"

input_path = Path("data/discovery_threshold_demo.csv")  # Default 18-document synthetic dataset.
drive_path = Path("/content/drive/MyDrive/YOUR_FILE.csv")  # For "drive".
output_root = Path("discovery_outputs")

text_column = "text"  # For "path".
id_column = "doc_id"  # For "path".

provider = "openrouter"
model_name = "google/gemini-3-flash-preview"
max_retries = 2

fuzzy_threshold = 0.9  # None disables fuzzy spelling matching.
seed = 0
n_codebook_examples = 3


In [8]:
import os
import getpass

api_key = os.getenv("OPENROUTER_API_KEY", "").strip()
if not api_key:
    api_key = getpass.getpass("Enter your OpenRouter API key: ").strip()
if not api_key:
    raise ValueError("An OpenRouter API key is required.")


In [9]:
import json
from copy import deepcopy
from html import escape

import pandas as pd
from IPython.display import display

import llm_tracker.discovery as discovery_module
from llm_tracker.config import AnalyzerConfig
from llm_tracker.discovery import (
    DISCOVERY_PROMPT,
    MERGE_PROMPT,
    REEVALUATE_PROMPT,
    LLMTrackerDiscoverer,
    apply_merge_edits,
    build_codebook,
    build_discovery_tables,
    codebook_to_frame,
    collapse_variants,
    filter_merged_dataframe,
    filter_merged_constructs,
    format_merged_constructs,
    frame_to_codebook,
    merge_constructs,
    reevaluate_merged_constructs,
    save_codebook,
)

def display_multiline_table(table, columns):
    """Display list items and multiline text on separate, HTML-safe lines."""
    preview = table.copy()
    for column in columns:
        preview[column] = preview[column].map(
            lambda value: "\n".join(value) if isinstance(value, list) else value
        )
    display(preview.style.format(escape="html").format(
        lambda value: escape(value).replace("\n", "<br>"),
        subset=columns,
        escape=None,
    ))


print("Discovery module:", discovery_module.__file__)
print("Working directory:", Path.cwd())


Discovery module: /Users/freymon.perez/Projects/GitHub/llm_tracker/src/llm_tracker/discovery.py
Working directory: /Users/freymon.perez/Projects/GitHub/llm_tracker


Run the next cell to open the data you selected. If uploading, choose **one
file**. If using Google Drive, approve access and check that `drive_path` gives
the address of your file or folder. Colab cannot open a file directly from an
address on your computer, such as `/Users/...` on a Mac; upload it or use Drive.

Practice texts are created only when you choose `input_source="demo"`. If the
practice file already exists, it is kept rather than replaced. Check the file
address printed below to confirm which texts will be sent to the LLM later.


In [10]:
if input_source not in {"demo", "threshold_demo", "path", "upload", "drive"}:
    raise ValueError('Choose input_source = "demo", "threshold_demo", "path", "upload", or "drive".')

if input_source == "upload":
    try:
        from google.colab import files
    except ImportError as error:
        raise RuntimeError('Upload is for Colab. On your computer, choose "path" and set input_path.') from error
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Choose exactly one file, then rerun this cell.")
    input_path = Path(next(iter(uploaded))).resolve()
elif input_source == "drive":
    try:
        from google.colab import drive
    except ImportError as error:
        raise RuntimeError('Drive mounting is for Colab. On your computer, choose "path" and use your local Drive folder.') from error
    drive.mount("/content/drive")
    input_path = drive_path.expanduser().resolve()
    if not input_path.is_relative_to(Path("/content/drive")):
        raise ValueError("Set drive_path to a file or folder under /content/drive/MyDrive/.")

if input_source == "threshold_demo":
    input_path = Path("data/discovery_threshold_demo.csv")
    text_column, id_column = "text", "doc_id"
elif input_source == "demo":
    input_path = Path("data/discovery_demo.csv")
    text_column, id_column = "text", "doc_id"
    input_path.parent.mkdir(parents=True, exist_ok=True)
    if not input_path.exists():
        demo = pd.DataFrame([
            {"doc_id": "demo_1", "text": "If I make one mistake at work, everyone will think I am incompetent."},
            {"doc_id": "demo_2", "text": "I assume my friend is angry whenever she takes a while to reply."},
            {"doc_id": "demo_3", "text": "I felt anxious about the meeting, so I avoided opening the invitation."},
            {"doc_id": "demo_4", "text": "I wrote down the evidence for and against my fear, which helped me consider another explanation."},
            {"doc_id": "demo_5", "text": "I took one small step toward the task instead of putting it off again."},
            {"doc_id": "demo_6", "text": "The delivery contained three blue folders and a box of paper clips."},
        ])
        demo.to_csv(input_path, index=False)
    display(pd.read_csv(input_path))

input_path = input_path.expanduser().resolve()
if not input_path.exists():
    raise FileNotFoundError(f"Update input_path; no input exists at {input_path.resolve()}")

print("Input selected:", input_path)
output_root = output_root.expanduser().resolve()
print("Outputs will save to:", output_root)

discovery_dir = output_root / "discovery"
merge_dir = output_root / "merged"
edits_dir = output_root / "edited"
filtered_dir = output_root / "filtered"
reevaluation_dir = output_root / "reevaluated"
tables_dir = output_root / "final_tables"
codebook_dir = output_root / "codebooks"
output_root.mkdir(parents=True, exist_ok=True)


Input selected: /Users/freymon.perez/Projects/GitHub/llm_tracker/data/discovery_threshold_demo.csv
Outputs will save to: /Users/freymon.perez/Projects/GitHub/llm_tracker/discovery_outputs


## 2. Review the instructions for finding constructs

A prompt is the written instruction sent to the LLM. The next cell shows the
instructions for identifying constructs in your texts. Reading or displaying
these instructions does not send anything to the LLM service.

You can keep the standard instructions. If you edit them, leave `{framework}`
and `{document}` in place: the notebook fills these in with your chosen
framework and each document's text. If adding other curly brackets, write them
as `{{` and `}}` so they are not mistaken for places to insert information.
The short example below shows how to change the instructions and then restores
the standard version before you continue.


In [11]:
config_kwargs = {
    "provider": provider,
    "max_retries": max_retries,
    "temperature": 0.0,
}
if model_name is not None:
    config_kwargs["model_name"] = model_name
if api_key is not None:
    config_kwargs["api_key"] = str(api_key)

config = AnalyzerConfig(**config_kwargs)
discoverer = LLMTrackerDiscoverer(config=config)
print("Model:", config.model_name)
print(discoverer.discovery_prompt)


Model: google/gemini-3-flash-preview
You are an expert qualitative researcher analyzing a document through the lens of a specific theoretical framework.

Framework: {framework}

Read the document below and identify the constructs from this framework that are present in the text. Use your own judgement about what counts as a construct of this framework; you are not working from a fixed list.

For each construct you find, provide:
- "construct": a short name for the construct
- "description": one sentence describing the construct as it appears here
- "quotes": a list of verbatim quotes from the document that evidence it

Only report constructs genuinely grounded in the text. If none are present, return an empty list.
Spell out construct names and descriptions in full. Do not use acronyms such as
CBT; write cognitive behavioral therapy instead.

Respond ONLY with a JSON object of the form:
{{"constructs": [{{"construct": "...", "description": "...", "quotes": ["..."]}}]}}

Document:
{docu

In [12]:
# Replace the prompt with a simple message so we can see that it changed.
discoverer.discovery_prompt = "We changed the prompt completely!"
print(discoverer.discovery_prompt)

# Change it back to the default prompt before running discovery.
discoverer.discovery_prompt = DISCOVERY_PROMPT
assert discoverer.discovery_prompt == DISCOVERY_PROMPT


We changed the prompt completely!


## 3. Discover constructs in the documents

This step sends your texts to the LLM, usually one request per document.
If a request fails, the notebook may try again, up to the number set in
`max_retries`. Running this cell again sends all the documents again and may
use more credits. Later steps reuse the results table rather than repeating
this analysis.

The results show one row for each supporting quote. If the model identifies a
construct but provides no quote, the row is kept with an empty quote field.
Documents analyzed successfully still count in the total used to calculate
percentages, even if the model finds no constructs in them. Documents that
could not be analyzed are excluded. If there are errors, the notebook stops
so you can review them before interpreting the findings.


In [13]:
discoveries, discovery_metadata, discovery_errors = discoverer.discover(
    path=input_path,
    framework=framework,
    text_column=text_column,
    id_column=id_column,
    output_dir=discovery_dir,
    save_format="both",
)

print("Successfully processed documents:", discoveries.attrs["total_documents"])
print("Discovery rows:", len(discoveries))
print("Failures:", len(discovery_errors))
display(discoveries)

if discovery_errors:
    display(pd.DataFrame([error.model_dump() for error in discovery_errors]))
    raise RuntimeError("Review the saved discovery errors before continuing. Rerunning discovery processes every document again.")
if discoveries.empty:
    raise RuntimeError("No constructs were discovered. Review the inputs, framework, and prompt before merging.")


Discovering across 18 document(s) from 1 file(s).

Discovering [1/18]: discovery_threshold_demo.csv::catastrophizing_1
  OK: 4 construct(s), 4 quote(s)
Discovering [2/18]: discovery_threshold_demo.csv::catastrophizing_2
  OK: 3 construct(s), 3 quote(s)
Discovering [3/18]: discovery_threshold_demo.csv::catastrophizing_3
  OK: 4 construct(s), 4 quote(s)
Discovering [4/18]: discovery_threshold_demo.csv::catastrophic_thinking_1
  OK: 3 construct(s), 4 quote(s)
Discovering [5/18]: discovery_threshold_demo.csv::catastrophic_thinking_2
  OK: 3 construct(s), 4 quote(s)
Discovering [6/18]: discovery_threshold_demo.csv::catastrophic_thinking_3
  OK: 3 construct(s), 4 quote(s)
Discovering [7/18]: discovery_threshold_demo.csv::experiential_avoidance_1
  OK: 3 construct(s), 3 quote(s)
Discovering [8/18]: discovery_threshold_demo.csv::experiential_avoidance_2
  OK: 2 construct(s), 2 quote(s)
Discovering [9/18]: discovery_threshold_demo.csv::experiential_avoidance_3
  OK: 2 construct(s), 2 quote(s)
D

,filename,doc_id,construct,description,quote
0,discovery_threshold_demo.csv,catastrophizing_1,catastrophizing,The individual engages in a cognitive distorti...,Catastrophizing: A typo in my report means my ...
1,discovery_threshold_demo.csv,catastrophizing_1,cognitive distortion,The text illustrates an irrational thought pat...,A typo in my report means my manager will fire...
2,discovery_threshold_demo.csv,catastrophizing_1,fortune telling,The individual predicts a series of negative f...,my manager will fire me and my career is over
3,discovery_threshold_demo.csv,catastrophizing_1,magnification,"The significance of a small mistake, a typo, i...",A typo in my report means my manager will fire...
4,discovery_threshold_demo.csv,catastrophizing_2,catastrophizing,A cognitive distortion where an individual ass...,Catastrophizing: A typo in my report means my ...
5,discovery_threshold_demo.csv,catastrophizing_2,cognitive distortion,An irrational or exaggerated thought pattern t...,A typo in my report means my manager will fire...
6,discovery_threshold_demo.csv,catastrophizing_2,fortune telling,A specific cognitive error involving the predi...,my manager will fire me and my career is over
7,discovery_threshold_demo.csv,catastrophizing_3,catastrophizing,The individual engages in a cognitive distorti...,A typo in my report means my manager will fire...
8,discovery_threshold_demo.csv,catastrophizing_3,cognitive distortion,The text illustrates an irrational thought pat...,A typo in my report means my manager will fire...
9,discovery_threshold_demo.csv,catastrophizing_3,all or nothing thinking,The individual views their career status in bi...,my career is over


## 4. Check for spelling differences in construct names

Slightly different names can refer to the same idea. This step checks for
differences such as "cognitive behavioral therapy" and "cognitive behavior
therapy" before the LLM groups constructs by meaning in the next step.

Names are made lowercase, and differences in spaces and separators such as
underscores are removed. The optional spelling check also combines names with
similar spellings. It compares spelling, not meaning. Set `fuzzy_threshold=None`
to turn off that extra check while keeping the basic cleanup. This step does
not send requests to the LLM.

In the table below, `count` is the number of different documents containing
each construct. A document is identified by its source file and document ID
together, so the same ID in two files refers to two documents. `instance_count`
counts supporting quotes, including repeated quote text; empty quotes are not
counted. Later tables use the name `document_count` instead of `count`.


In [14]:
normalized = collapse_variants(discoveries, fuzzy_threshold=fuzzy_threshold)
display(normalized[["construct", "count", "instance_count", "variants"]])
display(pd.DataFrame(normalized.attrs.get("name_matches", [])))


,construct,count,instance_count,variants
0,catastrophizing,6,6,[catastrophizing]
1,cognitive distortion,6,6,[cognitive distortion]
2,experiential avoidance,6,6,[experiential avoidance]
3,balanced thinking,5,5,[balanced thinking]
4,all or nothing thinking,4,7,[all or nothing thinking]
5,fortune telling,3,3,[fortune telling]
6,cognitive reappraisal,3,3,[cognitive reappraisal]
7,cognitive restructuring,3,3,[cognitive restructuring]
8,overgeneralization,3,3,[overgeneralization]
9,negative reinforcement,3,3,[negative reinforcement]


,original_name,construct_found
0,all or nothing thinking,all or nothing thinking
1,avoidance behavior,avoidance behavior
2,balanced thinking,balanced thinking
3,catastrophizing,catastrophizing
4,cognitive distortion,cognitive distortion
5,cognitive reappraisal,cognitive reappraisal
6,cognitive restructuring,cognitive restructuring
7,emotional avoidance,emotional avoidance
8,emotional regulation,emotional regulation
9,evidence testing,evidence testing


## 5. Group constructs by meaning

The LLM now looks for constructs that describe related ideas. It puts
them into groups and gives each group a name and definition. For example, it might
place "self hate" and "low self esteem" under "Negative self-perception".

This step is called merging. The example uses the standard grouping instructions,
`MERGE_PROMPT`. You can review or edit them, but keep `{framework}`, `{constructs}`,
and `{existing_note}` in place so the notebook can insert the information needed.

You can leave the settings below as they are for a first run. This table explains
the arguments (the information and choices supplied to the function) used by
`merge_constructs()` if you want to adjust them later. Names without quotation
marks, such as `discoveries` or `config`, refer to information prepared earlier
in the notebook; you do not need to type that information again.

| Setting | What it means |
|---|---|
| `df` | The results table to organize. Here, `discoveries` contains the constructs and supporting quotes found in your documents. Use this original table, not a shortened display or summary. |
| `config` | The model and connection settings already chosen above. Keep `config` here to use that same model for grouping. |
| `framework` | The research perspective used during discovery, such as emotions or cognitive processes. It guides which meanings the model considers related; normally keep the same framework here. |
| `merge_prompt` | The written instructions for grouping the constructs. |
| `fuzzy_threshold` | How similar two names' spellings must be before combining them, before the model groups them by meaning. The default, `0.92`, is a spelling-similarity setting, not a reliability statistic. A higher number is stricter; `None` turns this approximate spelling matching off, but basic name cleanup still occurs. |
| `seed` | Reusing this number puts the construct names in the same shuffled order. The LLM may still give different answers. |
| `batch` | `False` lets the model consider the full list together, which is preferred when it fits. `True` sends smaller portions sequentially and builds on the groups formed so far. This makes multiple model requests and can change the grouping and cost. |
| `batch_size` | The number of construct names in each portion when `batch=True`. The default is 150; this setting has no effect when `batch=False`, as used below. |
| `max_prompt_chars` | A safeguard limiting the length of the instructions and construct information sent in one request. `60000` means 60,000 text characters, not documents or constructs. An oversized request is stopped before it is sent. |
| `existing` | Optional earlier grouping to build on. It is not supplied below, so this walkthrough starts a new grouping. This is not a way to lock earlier groups against changes. |
| `total_documents` | The number of successfully analyzed documents, including those with no constructs found. This is filled in automatically; enter it yourself only if reopening saved results that no longer include it. |
| `output_dir` | The destination folder. Here, `merge_dir` is the folder prepared for the grouping results. Leave this argument out if you do not want this function to save files. |
| `save_format` | The results-table format: `"csv"` for spreadsheet software, `"json"` to preserve lists such as quotes, or `"both"`. Supporting JSON files, including the group arrangement and information needed to reload results, are also saved when `output_dir` is supplied, even with `"csv"`. |

The function returns both the groups (`merged`) and their detailed results table
(`merge_table`). **Rerunning this cell sends model requests again**; it does not
simply reopen saved results, and API charges may apply.


In [15]:
merged, merge_table = merge_constructs(
    df=discoveries,
    config=config,
    framework=framework,
    merge_prompt=MERGE_PROMPT,
    fuzzy_threshold=fuzzy_threshold,
    seed=seed,
    batch=False,
    max_prompt_chars=60000,
    output_dir=merge_dir,
    save_format="both",
)

display_multiline_table(format_merged_constructs(merge_table, n_quotes=3, seed=seed), ["quotes"])


Collapsed 17 discovered name(s) into 17 distinct construct(s).
Saved merge output to /Users/freymon.perez/Projects/GitHub/llm_tracker/discovery_outputs/merged


,merged_construct,definition,status,construct_found,document_count,instance_count,document_frequency_percent,total_documents,prototypicality,quotes,document_frequency
0,Catastrophizing,A cognitive distortion involving the assumption that the worst possible outcome will occur or that a situation is far worse than it actually is.,merged,catastrophizing,6,6,33.333333,18,1.000000,Catastrophizing: A typo in my report means my manager will fire me and my career is over.A typo in my report means my manager will fire me and my career is over.Missing one bus means the whole day is ruined and I will never succeed.,33.3%
1,Catastrophizing,A cognitive distortion involving the assumption that the worst possible outcome will occur or that a situation is far worse than it actually is.,merged,magnification,1,1,5.555556,18,0.900000,A typo in my report means my manager will fire me and my career is over.,5.6%
2,Catastrophizing,A cognitive distortion involving the assumption that the worst possible outcome will occur or that a situation is far worse than it actually is.,merged,fortune telling,3,3,16.666667,18,0.850000,my manager will fire me and my career is overmy manager will fire me,16.7%
3,Cognitive Distortion,Irrational or biased thought patterns that misinterpret reality and contribute to emotional distress.,merged,cognitive distortion,6,6,33.333333,18,1.000000,A typo in my report means my manager will fire me and my career is over.'I always fail'I always fail,33.3%
4,Cognitive Distortion,Irrational or biased thought patterns that misinterpret reality and contribute to emotional distress.,merged,all or nothing thinking,4,7,22.222222,18,0.850000,my career is overthe whole day is ruinedI will never succeed,22.2%
5,Cognitive Distortion,Irrational or biased thought patterns that misinterpret reality and contribute to emotional distress.,merged,overgeneralization,3,3,16.666667,18,0.850000,I will never succeedMissing one bus means... I will never succeed.,16.7%
6,Cognitive Restructuring,"A therapeutic process of identifying, challenging, and replacing irrational or maladaptive thought patterns with more accurate and functional beliefs.",merged,cognitive restructuring,3,3,16.666667,18,1.000000,I question the thought 'I always fail' and replace it with a more balanced thought.Cognitive restructuring: I question the thought 'I always fail' and replace it with a more balanced thought.,16.7%
7,Cognitive Restructuring,"A therapeutic process of identifying, challenging, and replacing irrational or maladaptive thought patterns with more accurate and functional beliefs.",merged,cognitive reappraisal,3,3,16.666667,18,0.900000,"Cognitive reappraisal: I list evidence for and against my worry, then consider a balanced explanation.",16.7%
8,Cognitive Restructuring,"A therapeutic process of identifying, challenging, and replacing irrational or maladaptive thought patterns with more accurate and functional beliefs.",merged,evidence testing,3,3,16.666667,18,0.850000,I list evidence for and against my worry,16.7%
9,Cognitive Restructuring,"A therapeutic process of identifying, challenging, and replacing irrational or maladaptive thought patterns with more accurate and functional beliefs.",merged,balanced thinking,5,5,27.777778,18,0.800000,consider a balanced explanationreplace it with a more balanced thought,27.8%


Each row shows one **discovered construct** and the **group** it belongs to.
The table calls these merged groups. A group can appear on several rows because
it contains several discovered constructs.

| Column | In plain language |
|---|---|
| `merged_construct` | The name of the larger group, such as "Negative self-perception". |
| `definition` | What that group means. |
| `construct_found` | The name originally found in the documents, such as "self hate", after spelling cleanup. |
| `document_count` | How many different documents contained this discovered construct. Finding it three times in one document still counts as one document. |
| `instance_count` | How many supporting quotes were found, excluding empty quotes. Repeated quote text is counted each time it was found, even across different documents. |
| `document_frequency` | How common it was: `1.7%` means that percentage of successfully processed documents contained it. |
| `document_frequency_percent` | The same percentage as a number, useful for calculations. |
| `total_documents` | All successfully processed documents, including documents where nothing was found. Failed documents are excluded. |
| `prototypicality` | How well this discovered construct represents its merged group. A score near **1** means a strong, typical example; a score near **0** means a less typical member. This is **not** the confidence that a quote was detected correctly, or how often it appeared. |
| `quotes` | Supporting text. The table shown above displays up to three quotes per row, each on a separate line. The full results keep all quotes, with identical text listed only once per row. |
| `status` | `merged`: assigned to a group. `standalone`: kept on its own. `unplaced`: not assigned by the model, but still kept in the results so it is not lost. |

An empty prototypicality score means no score is available; it does not mean zero.
Later steps use `merge_table`, the full results table. The shortened display is
only a reading aid; it does not replace or remove information from that table.
Here, `batch=True` means sending smaller requests one after another, not using
a provider's separate service for submitting jobs to be processed later.


### Choose minimum document counts and group-fit scores

You may want to set aside constructs found in only a few documents or judged
less typical of their group. The next cell separates them into two tables:

- **Kept:** constructs found in at least **2 different documents** and with a
  **prototypicality score of at least 0.3**. This score describes how well an
  idea fits its group, from 0 (less typical) to 1 (very typical).
- **Removed:** everything that did not meet one or both minimums. A construct
  without a score is also set aside while the score minimum is turned on.

A construct must meet both minimums to be kept; values exactly at the minimums
qualify. The document minimum counts different documents, not percentages or
quotes. Each discovered construct is checked separately, rather than the group
as a whole. Change the two numbers below to choose your minimums. Set either to
`None` to turn off that check.

Both tables are saved in your main results folder, `output_root`, as CSV files
for spreadsheets and JSON files for later use in the notebook. Their names start
with `filtered_merge_table_` and `removed_merge_table_`.
Nothing is deleted from the original merged table. You can inspect the removed
rows and rerun this cell with different minimums without sending another request
to the LLM. Later steps use the kept rows. With a small practice dataset, these
minimums may remove everything; lower them or turn them off to continue.


In [16]:
min_document_count = 2
min_prototypicality = 0.3

threshold_table, removed_table = filter_merged_dataframe(
    table=merge_table,
    min_document_count=min_document_count,
    min_prototypicality=min_prototypicality,
    output_dir=output_root,
    save_format="both",
)
print("Kept constructs:", len(threshold_table))
display_multiline_table(format_merged_constructs(threshold_table, n_quotes=3, seed=seed), ["quotes"])
print("Removed constructs:", len(removed_table))
display_multiline_table(format_merged_constructs(removed_table, n_quotes=3, seed=seed), ["quotes"])
print("Both tables saved to:", output_root)

if threshold_table.empty:
    raise RuntimeError("No constructs met both minimums. Lower the values above or set them to None, then rerun this cell.")


Saved retained and removed merge tables to /Users/freymon.perez/Projects/GitHub/llm_tracker/discovery_outputs
Kept constructs: 13


,merged_construct,definition,status,construct_found,document_count,instance_count,document_frequency_percent,total_documents,prototypicality,quotes,document_frequency
0,Catastrophizing,A cognitive distortion involving the assumption that the worst possible outcome will occur or that a situation is far worse than it actually is.,merged,catastrophizing,6,6,33.333333,18,1.000000,Catastrophizing: A typo in my report means my manager will fire me and my career is over.A typo in my report means my manager will fire me and my career is over.Missing one bus means the whole day is ruined and I will never succeed.,33.3%
2,Catastrophizing,A cognitive distortion involving the assumption that the worst possible outcome will occur or that a situation is far worse than it actually is.,merged,fortune telling,3,3,16.666667,18,0.850000,my manager will fire me and my career is overmy manager will fire me,16.7%
3,Cognitive Distortion,Irrational or biased thought patterns that misinterpret reality and contribute to emotional distress.,merged,cognitive distortion,6,6,33.333333,18,1.000000,A typo in my report means my manager will fire me and my career is over.'I always fail'I always fail,33.3%
4,Cognitive Distortion,Irrational or biased thought patterns that misinterpret reality and contribute to emotional distress.,merged,all or nothing thinking,4,7,22.222222,18,0.850000,my career is overthe whole day is ruinedI will never succeed,22.2%
5,Cognitive Distortion,Irrational or biased thought patterns that misinterpret reality and contribute to emotional distress.,merged,overgeneralization,3,3,16.666667,18,0.850000,I will never succeedMissing one bus means... I will never succeed.,16.7%
6,Cognitive Restructuring,"A therapeutic process of identifying, challenging, and replacing irrational or maladaptive thought patterns with more accurate and functional beliefs.",merged,cognitive restructuring,3,3,16.666667,18,1.000000,I question the thought 'I always fail' and replace it with a more balanced thought.Cognitive restructuring: I question the thought 'I always fail' and replace it with a more balanced thought.,16.7%
7,Cognitive Restructuring,"A therapeutic process of identifying, challenging, and replacing irrational or maladaptive thought patterns with more accurate and functional beliefs.",merged,cognitive reappraisal,3,3,16.666667,18,0.900000,"Cognitive reappraisal: I list evidence for and against my worry, then consider a balanced explanation.",16.7%
8,Cognitive Restructuring,"A therapeutic process of identifying, challenging, and replacing irrational or maladaptive thought patterns with more accurate and functional beliefs.",merged,evidence testing,3,3,16.666667,18,0.850000,I list evidence for and against my worry,16.7%
9,Cognitive Restructuring,"A therapeutic process of identifying, challenging, and replacing irrational or maladaptive thought patterns with more accurate and functional beliefs.",merged,balanced thinking,5,5,27.777778,18,0.800000,consider a balanced explanationreplace it with a more balanced thought,27.8%
10,Experiential Avoidance,"The tendency to avoid, suppress, or escape from uncomfortable internal experiences, often through behaviors that provide short-term relief but lead to long-term impairment.",merged,experiential avoidance,6,6,33.333333,18,1.000000,I cancel plans whenever I feel anxious so I do not have to face discomfort.I postpone opening the bill because I do not want to feel worried.,33.3%


Removed constructs: 4


,merged_construct,definition,status,construct_found,document_count,instance_count,document_frequency_percent,total_documents,prototypicality,quotes,document_frequency
1,Catastrophizing,A cognitive distortion involving the assumption that the worst possible outcome will occur or that a situation is far worse than it actually is.,merged,magnification,1,1,5.555556,18,0.900000,A typo in my report means my manager will fire me and my career is over.,5.6%
11,Experiential Avoidance,"The tendency to avoid, suppress, or escape from uncomfortable internal experiences, often through behaviors that provide short-term relief but lead to long-term impairment.",merged,emotional avoidance,1,1,5.555556,18,0.950000,I do not want to feel worried,5.6%
13,Experiential Avoidance,"The tendency to avoid, suppress, or escape from uncomfortable internal experiences, often through behaviors that provide short-term relief but lead to long-term impairment.",merged,safety behavior,1,1,5.555556,18,0.850000,I cancel plans whenever I feel anxious so I do not have to face discomfort.,5.6%
15,Experiential Avoidance,"The tendency to avoid, suppress, or escape from uncomfortable internal experiences, often through behaviors that provide short-term relief but lead to long-term impairment.",merged,maladaptive coping,1,1,5.555556,18,0.750000,I postpone opening the bill,5.6%


Both tables saved to: /Users/freymon.perez/Projects/GitHub/llm_tracker/discovery_outputs


## 6. Review and edit the groups

The next cell shows each group name followed by the constructs assigned to it.
This arrangement is called a dictionary in Python. A construct belonging to a
group may also be called a constituent. The grouping file is saved with a name
such as `merged_constructs_groups_<timestamp>.json`, where `<timestamp>` is
replaced by the date and time.

To revise the groups, copy the displayed arrangement into `edited_grouping` in
the following cell, replacing `deepcopy(grouping)`. You can rename a group,
move a construct to another group's list, or remove a construct from the lists.
Keep the quotation marks, commas, and brackets, and list each construct only
once. If you do not want to make changes, leave the code as it is.

This step applies your edits without sending requests to the LLM. The settings
in `apply_merge_edits()` mean:

| Setting | What it means |
|---|---|
| `table` | The detailed results kept in the previous step, here `threshold_table`. Use the underlying table, not its formatted preview. Counts and quotes stay with each discovered construct when you change its group. |
| `grouping` | Your revised arrangement, here `edited_grouping`: each group name followed by a list of the discovered constructs it should contain. Use construct names from the displayed arrangement; this step cannot add newly discovered constructs. |
| `definitions` | Optional definitions you write yourself, matched to the revised group names. For example, `definitions={"Low-arousal positive affect": "Pleasant feelings with low activation, including calmness, contentment, and relief."}`. Remove the initial `#` from the example argument below to use it. A rename alone keeps the old definition where possible; it does not automatically rewrite it. A new group mixing several source groups has a blank definition unless you supply one. |
| `total_documents` | The number of successfully analyzed documents, including those with no constructs found. Normally carried forward automatically. Removing constructs does not reduce this number or change the basis for percentages. |
| `output_dir` | The destination folder, here `edits_dir`, for the updated results and grouping files. New timestamped files are saved rather than overwriting earlier ones. Leave this argument out to work without saving files at this step. |
| `save_format` | The updated table's format: `"csv"` for spreadsheet software, `"json"` to preserve lists, or `"both"`. Supporting JSON files for the groups and editing context are saved as well when a destination folder is supplied. |

Leaving a construct out of every group removes it from the next steps. Listing
it more than once causes an error. Editing groups does not repeat the original
analysis of your texts. The function returns the revised groups (`edited_merged`)
and table (`edited_table`), leaving the input table unchanged. It does not
automatically rewrite definitions or obtain new model judgments; the later
reevaluation step is available for model-assisted revisions.


In [17]:
grouping = {
    name: group["construct_found"].tolist()
    for name, group in threshold_table.groupby("merged_construct", sort=False)
}
print(json.dumps(grouping, indent=2, ensure_ascii=False))


{
  "Catastrophizing": [
    "catastrophizing",
    "fortune telling"
  ],
  "Cognitive Distortion": [
    "cognitive distortion",
    "all or nothing thinking",
    "overgeneralization"
  ],
  "Cognitive Restructuring": [
    "cognitive restructuring",
    "cognitive reappraisal",
    "evidence testing",
    "balanced thinking"
  ],
  "Experiential Avoidance": [
    "experiential avoidance",
    "avoidance behavior",
    "negative reinforcement",
    "emotional regulation"
  ]
}


In [18]:
# Replace this assignment with your edited dictionary, or edit the copy below.
edited_grouping = deepcopy(grouping)

# Examples, using names that actually occur in the printed dictionary:
# edited_grouping["New group name"] = edited_grouping.pop("Old group name")
# edited_grouping["Group name"].remove("constituent to remove")

edited_merged, edited_table = apply_merge_edits(
    table=threshold_table,
    grouping=edited_grouping,
    # definitions={"New group name": "Your revised definition."},
    output_dir=edits_dir,
    save_format="both",
)
display_multiline_table(format_merged_constructs(edited_table, n_quotes=3, seed=seed), ["quotes"])


Saved merge output to /Users/freymon.perez/Projects/GitHub/llm_tracker/discovery_outputs/edited


,merged_construct,definition,status,construct_found,document_count,instance_count,document_frequency_percent,total_documents,prototypicality,quotes,document_frequency
0,Catastrophizing,A cognitive distortion involving the assumption that the worst possible outcome will occur or that a situation is far worse than it actually is.,merged,catastrophizing,6,6,33.333333,18,1.000000,Catastrophizing: A typo in my report means my manager will fire me and my career is over.A typo in my report means my manager will fire me and my career is over.Missing one bus means the whole day is ruined and I will never succeed.,33.3%
1,Catastrophizing,A cognitive distortion involving the assumption that the worst possible outcome will occur or that a situation is far worse than it actually is.,merged,fortune telling,3,3,16.666667,18,0.850000,my manager will fire me and my career is overmy manager will fire me,16.7%
2,Cognitive Distortion,Irrational or biased thought patterns that misinterpret reality and contribute to emotional distress.,merged,cognitive distortion,6,6,33.333333,18,1.000000,A typo in my report means my manager will fire me and my career is over.'I always fail'I always fail,33.3%
3,Cognitive Distortion,Irrational or biased thought patterns that misinterpret reality and contribute to emotional distress.,merged,all or nothing thinking,4,7,22.222222,18,0.850000,my career is overthe whole day is ruinedI will never succeed,22.2%
4,Cognitive Distortion,Irrational or biased thought patterns that misinterpret reality and contribute to emotional distress.,merged,overgeneralization,3,3,16.666667,18,0.850000,I will never succeedMissing one bus means... I will never succeed.,16.7%
5,Cognitive Restructuring,"A therapeutic process of identifying, challenging, and replacing irrational or maladaptive thought patterns with more accurate and functional beliefs.",merged,cognitive restructuring,3,3,16.666667,18,1.000000,I question the thought 'I always fail' and replace it with a more balanced thought.Cognitive restructuring: I question the thought 'I always fail' and replace it with a more balanced thought.,16.7%
6,Cognitive Restructuring,"A therapeutic process of identifying, challenging, and replacing irrational or maladaptive thought patterns with more accurate and functional beliefs.",merged,cognitive reappraisal,3,3,16.666667,18,0.900000,"Cognitive reappraisal: I list evidence for and against my worry, then consider a balanced explanation.",16.7%
7,Cognitive Restructuring,"A therapeutic process of identifying, challenging, and replacing irrational or maladaptive thought patterns with more accurate and functional beliefs.",merged,evidence testing,3,3,16.666667,18,0.850000,I list evidence for and against my worry,16.7%
8,Cognitive Restructuring,"A therapeutic process of identifying, challenging, and replacing irrational or maladaptive thought patterns with more accurate and functional beliefs.",merged,balanced thinking,5,5,27.777778,18,0.800000,consider a balanced explanationreplace it with a more balanced thought,27.8%
9,Experiential Avoidance,"The tendency to avoid, suppress, or escape from uncomfortable internal experiences, often through behaviors that provide short-term relief but lead to long-term impairment.",merged,experiential avoidance,6,6,33.333333,18,1.000000,I cancel plans whenever I feel anxious so I do not have to face discomfort.I postpone opening the bill because I do not want to feel worried.,33.3%


Each construct keeps its counts and quotes. The total number of analyzed
documents used to calculate percentages does not change.

Existing definitions are kept where the original group can be identified. If
you create a group that combines constructs from several groups, you may need
to supply a new definition. Review definitions whenever you change membership.

A construct moved to a different group loses its old prototypicality score,
because that score described how well it fit the previous group. Renaming a
group without changing its members keeps the scores. The optional review step
below can update names and definitions, but does not calculate new scores.


## 7. Optionally remove more constructs after your edits

The minimums above have already been applied. This step lets you remove named
constructs or whole groups after reviewing them. The settings below make no
additional cuts unless you change them.

Use `remove_constructs` for individual discovered constructs and
`remove_merged_constructs` for whole groups. Copy names from your results;
capitalization and separators such as underscores are ignored when matching
them. A name not found in the results causes an error.

You can also set additional minimums using `review_min_document_count`,
`review_min_instance_count` (number of supporting quotes), and
`review_min_prototypicality`. A construct must meet every minimum you set;
values exactly at a minimum qualify. A score minimum removes constructs without
scores, including those kept on their own, left unassigned, or moved between
groups. Leave `review_min_prototypicality=None` to avoid removing them for
missing scores. This step sends no requests to the LLM and does not change the
total number of analyzed documents used to calculate percentages.


In [19]:
remove_constructs = []       # Constituent names to remove.
remove_merged_constructs = []  # Whole merged groups to remove.
review_min_document_count = None
review_min_instance_count = None
review_min_prototypicality = None

filtered_merged, filtered_table = filter_merged_constructs(
    table=edited_table,
    remove_constructs=remove_constructs,
    remove_merged_constructs=remove_merged_constructs,
    min_document_count=review_min_document_count,
    min_instance_count=review_min_instance_count,
    min_prototypicality=review_min_prototypicality,
    output_dir=filtered_dir,
    save_format="both",
)
final_merged, final_table = filtered_merged, filtered_table
display_multiline_table(format_merged_constructs(filtered_table, n_quotes=3, seed=seed), ["quotes"])
print("Constituents remaining:", len(filtered_table))
print("Removed members recorded:", len(filtered_table.attrs.get("removed_constituents", [])))

if filtered_table.empty:
    raise RuntimeError("All constituents were removed. Relax the filters or revise the grouping before building a codebook.")


Saved merge output to /Users/freymon.perez/Projects/GitHub/llm_tracker/discovery_outputs/filtered


,merged_construct,definition,status,construct_found,document_count,instance_count,document_frequency_percent,total_documents,prototypicality,quotes,document_frequency
0,Catastrophizing,A cognitive distortion involving the assumption that the worst possible outcome will occur or that a situation is far worse than it actually is.,merged,catastrophizing,6,6,33.333333,18,1.000000,Catastrophizing: A typo in my report means my manager will fire me and my career is over.A typo in my report means my manager will fire me and my career is over.Missing one bus means the whole day is ruined and I will never succeed.,33.3%
1,Catastrophizing,A cognitive distortion involving the assumption that the worst possible outcome will occur or that a situation is far worse than it actually is.,merged,fortune telling,3,3,16.666667,18,0.850000,my manager will fire me and my career is overmy manager will fire me,16.7%
2,Cognitive Distortion,Irrational or biased thought patterns that misinterpret reality and contribute to emotional distress.,merged,cognitive distortion,6,6,33.333333,18,1.000000,A typo in my report means my manager will fire me and my career is over.'I always fail'I always fail,33.3%
3,Cognitive Distortion,Irrational or biased thought patterns that misinterpret reality and contribute to emotional distress.,merged,all or nothing thinking,4,7,22.222222,18,0.850000,my career is overthe whole day is ruinedI will never succeed,22.2%
4,Cognitive Distortion,Irrational or biased thought patterns that misinterpret reality and contribute to emotional distress.,merged,overgeneralization,3,3,16.666667,18,0.850000,I will never succeedMissing one bus means... I will never succeed.,16.7%
5,Cognitive Restructuring,"A therapeutic process of identifying, challenging, and replacing irrational or maladaptive thought patterns with more accurate and functional beliefs.",merged,cognitive restructuring,3,3,16.666667,18,1.000000,I question the thought 'I always fail' and replace it with a more balanced thought.Cognitive restructuring: I question the thought 'I always fail' and replace it with a more balanced thought.,16.7%
6,Cognitive Restructuring,"A therapeutic process of identifying, challenging, and replacing irrational or maladaptive thought patterns with more accurate and functional beliefs.",merged,cognitive reappraisal,3,3,16.666667,18,0.900000,"Cognitive reappraisal: I list evidence for and against my worry, then consider a balanced explanation.",16.7%
7,Cognitive Restructuring,"A therapeutic process of identifying, challenging, and replacing irrational or maladaptive thought patterns with more accurate and functional beliefs.",merged,evidence testing,3,3,16.666667,18,0.850000,I list evidence for and against my worry,16.7%
8,Cognitive Restructuring,"A therapeutic process of identifying, challenging, and replacing irrational or maladaptive thought patterns with more accurate and functional beliefs.",merged,balanced thinking,5,5,27.777778,18,0.800000,consider a balanced explanationreplace it with a more balanced thought,27.8%
9,Experiential Avoidance,"The tendency to avoid, suppress, or escape from uncomfortable internal experiences, often through behaviors that provide short-term relief but lead to long-term impairment.",merged,experiential avoidance,6,6,33.333333,18,1.000000,I cancel plans whenever I feel anxious so I do not have to face discomfort.I postpone opening the bill because I do not want to feel worried.,33.3%


Constituents remaining: 13
Removed members recorded: 4


## 8. Optional: ask the LLM to review group names and definitions

Run the function call below when you want the LLM to review the group names and definitions. In scripted workflows, you may gate the call on a trigger. Running this cell sends an additional LLM request and may use additional credits.

The model is shown the earlier groups, the revised groups, and constructs that were removed. It can change names and definitions only. It cannot move constructs between groups or bring back removed ones. Quotes, counts, the total used for percentages, and all available or missing scores stay unchanged.

The standard instructions are in `REEVALUATE_PROMPT`. You can leave them as they are. If editing them, keep `{framework}`, `{previous}`, `{removed}`, and `{current}`, along with the requested answer format, so the notebook can fill in the information and read the reply. `max_prompt_chars` limits the length of the instructions and included results before they are sent.


In [20]:
# Print this to inspect the reevaluation instructions without an API call:
# print(REEVALUATE_PROMPT)

# This sends an additional LLM call.
final_merged, final_table = reevaluate_merged_constructs(
    table=filtered_table,
    config=config,
    framework=framework,
    reevaluation_prompt=REEVALUATE_PROMPT,
    max_prompt_chars=60000,
    output_dir=reevaluation_dir,
    save_format="both",
)

display_multiline_table(format_merged_constructs(final_table, n_quotes=3, seed=seed), ["quotes"])


Saved merge output to /Users/freymon.perez/Projects/GitHub/llm_tracker/discovery_outputs/reevaluated


,merged_construct,definition,status,construct_found,document_count,instance_count,document_frequency_percent,total_documents,prototypicality,quotes,document_frequency
0,Catastrophizing,A cognitive distortion characterized by predicting the worst possible outcome and treating that outcome as an inevitable or unmanageable certainty.,merged,catastrophizing,6,6,33.333333,18,1.000000,Catastrophizing: A typo in my report means my manager will fire me and my career is over.A typo in my report means my manager will fire me and my career is over.Missing one bus means the whole day is ruined and I will never succeed.,33.3%
1,Catastrophizing,A cognitive distortion characterized by predicting the worst possible outcome and treating that outcome as an inevitable or unmanageable certainty.,merged,fortune telling,3,3,16.666667,18,0.850000,my manager will fire me and my career is overmy manager will fire me,16.7%
2,Cognitive Distortion,"Biased or irrational thought patterns, such as overgeneralization and all-or-nothing thinking, that misinterpret reality and reinforce negative emotional states.",merged,cognitive distortion,6,6,33.333333,18,1.000000,A typo in my report means my manager will fire me and my career is over.'I always fail'I always fail,33.3%
3,Cognitive Distortion,"Biased or irrational thought patterns, such as overgeneralization and all-or-nothing thinking, that misinterpret reality and reinforce negative emotional states.",merged,all or nothing thinking,4,7,22.222222,18,0.850000,my career is overthe whole day is ruinedI will never succeed,22.2%
4,Cognitive Distortion,"Biased or irrational thought patterns, such as overgeneralization and all-or-nothing thinking, that misinterpret reality and reinforce negative emotional states.",merged,overgeneralization,3,3,16.666667,18,0.850000,I will never succeedMissing one bus means... I will never succeed.,16.7%
5,Cognitive Restructuring,"A therapeutic technique involving the identification, evaluation, and modification of maladaptive thoughts to foster more balanced and evidence-based perspectives.",merged,cognitive restructuring,3,3,16.666667,18,1.000000,I question the thought 'I always fail' and replace it with a more balanced thought.Cognitive restructuring: I question the thought 'I always fail' and replace it with a more balanced thought.,16.7%
6,Cognitive Restructuring,"A therapeutic technique involving the identification, evaluation, and modification of maladaptive thoughts to foster more balanced and evidence-based perspectives.",merged,cognitive reappraisal,3,3,16.666667,18,0.900000,"Cognitive reappraisal: I list evidence for and against my worry, then consider a balanced explanation.",16.7%
7,Cognitive Restructuring,"A therapeutic technique involving the identification, evaluation, and modification of maladaptive thoughts to foster more balanced and evidence-based perspectives.",merged,evidence testing,3,3,16.666667,18,0.850000,I list evidence for and against my worry,16.7%
8,Cognitive Restructuring,"A therapeutic technique involving the identification, evaluation, and modification of maladaptive thoughts to foster more balanced and evidence-based perspectives.",merged,balanced thinking,5,5,27.777778,18,0.800000,consider a balanced explanationreplace it with a more balanced thought,27.8%
9,Experiential Avoidance,"The process of attempting to escape or suppress unpleasant internal experiences, such as anxious thoughts or distressing emotions, through behavioral or psychological withdrawal.",merged,experiential avoidance,6,6,33.333333,18,1.000000,I cancel plans whenever I feel anxious so I do not have to face discomfort.I postpone opening the bill because I do not want to feel worried.,33.3%


## 9. Save all quotes and a construct summary

This step saves two tables without sending requests to the LLM:

1. **All quotes:** one row for each time a supporting quote was found, with its
   source file, document ID, original construct name, name after spelling
   cleanup, and final group. The same text found in different documents appears
   on separate rows.
2. **Construct summary:** one row per final group, with the columns below.
   There is no extra row combining all groups into an overall result.

The quote table keeps the original quotes even for constructs you removed during
review. Its `retained` column shows whether each construct was kept in the final
groups. Removed constructs have no final group listed. The summary table
contains only the groups you kept after editing and filtering.

| Summary column | What it tells you |
|---|---|
| Construct | The final merged group name. |
| Definition | What that group means. |
| Construct found % (N) | The number of different discovered constructs in this group, and their percentage of all constructs originally found after spelling cleanup. |
| Document count % (N) | The number of different documents containing this group, and their percentage of all successfully analyzed documents. A document with several constructs from the same group counts only once for that group. |
| Instance Count % (N) | The number of supporting quotes for this group, and their percentage of all supporting quotes originally found. Repeated quote text is counted each time it was found. |
| Prototypicality mean (SD) | The mean and standard deviation of the available scores for constructs in this group. Smaller SD means more similar scores. A single score has no SD; missing scores are left out, not treated as zero. |
| Examples | All supporting quote texts for this group, with identical text listed only once. This is not limited to three. |
| Constituents found | The discovered constructs kept in this group, displayed one per line without list brackets. |

For example, `50.0% (2/4)` means two out of four. Percentages always use the
original totals, even after you remove constructs. A document can contain
constructs from more than one group, so document percentages across groups may
add up to more than 100%.

The next cell uses `build_discovery_tables()` to prepare these tables from the
**original discovery results** (`discoveries`) and your **final reviewed results**
(`final_table`). Keep the same `fuzzy_threshold` setting used earlier so names
match consistently. `output_dir` selects the results folder. `save_format="both"`
saves CSV files for spreadsheets and JSON files that keep lists together. An
additional JSON file records the original totals used to calculate percentages.


In [21]:
all_quotes, construct_summary = build_discovery_tables(
    discoveries=discoveries,
    merged_table=final_table,
    fuzzy_threshold=fuzzy_threshold,
    output_dir=tables_dir,
    save_format="both",
)

# Both tables are saved in full; pandas may shorten the on-screen preview.
with pd.option_context("display.max_colwidth", None):
    display(all_quotes)
    display_multiline_table(construct_summary, ["Examples", "Constituents found"])
print("Final tables saved to:", tables_dir.resolve())


Saved all original quote occurrences and final construct summary to /Users/freymon.perez/Projects/GitHub/llm_tracker/discovery_outputs/final_tables


,filename,doc_id,merged_construct,construct_found,original_construct,description,quote,retained
0,discovery_threshold_demo.csv,catastrophizing_1,Catastrophizing,catastrophizing,catastrophizing,The individual engages in a cognitive distortion where they assume the worst possible outcome will occur from a relatively minor event.,Catastrophizing: A typo in my report means my manager will fire me and my career is over.,True
1,discovery_threshold_demo.csv,catastrophizing_1,Cognitive Distortion,cognitive distortion,cognitive distortion,"The text illustrates an irrational thought pattern that misinterprets reality, specifically through the lens of an exaggerated negative consequence.",A typo in my report means my manager will fire me and my career is over.,True
2,discovery_threshold_demo.csv,catastrophizing_1,Catastrophizing,fortune telling,fortune telling,"The individual predicts a series of negative future events, such as being fired and their career ending, without sufficient evidence.",my manager will fire me and my career is over,True
3,discovery_threshold_demo.csv,catastrophizing_1,NaN,magnification,magnification,"The significance of a small mistake, a typo, is disproportionately enlarged to represent a life-altering failure.",A typo in my report means my manager will fire me and my career is over.,False
4,discovery_threshold_demo.csv,catastrophizing_2,Catastrophizing,catastrophizing,catastrophizing,A cognitive distortion where an individual assumes the worst possible outcome will occur from a relatively minor event.,Catastrophizing: A typo in my report means my manager will fire me and my career is over.,True
5,discovery_threshold_demo.csv,catastrophizing_2,Cognitive Distortion,cognitive distortion,cognitive distortion,An irrational or exaggerated thought pattern that reinforces negative emotions and inaccurate perceptions of reality.,A typo in my report means my manager will fire me and my career is over.,True
6,discovery_threshold_demo.csv,catastrophizing_2,Catastrophizing,fortune telling,fortune telling,A specific cognitive error involving the prediction of a negative future outcome without sufficient evidence.,my manager will fire me and my career is over,True
7,discovery_threshold_demo.csv,catastrophizing_3,Catastrophizing,catastrophizing,catastrophizing,The individual engages in a cognitive distortion where they assume the worst possible outcome will occur from a minor event.,A typo in my report means my manager will fire me and my career is over.,True
8,discovery_threshold_demo.csv,catastrophizing_3,Cognitive Distortion,cognitive distortion,cognitive distortion,The text illustrates an irrational thought pattern that exaggerates the negative consequences of a small mistake.,A typo in my report means my manager will fire me and my career is over.,True
9,discovery_threshold_demo.csv,catastrophizing_3,Cognitive Distortion,all or nothing thinking,all or nothing thinking,"The individual views their career status in binary terms, where a single error equates to total professional failure.",my career is over,True


,Construct,Definition,Construct found % (N),Document count % (N),Instance Count % (N),Prototypicality mean (SD),Examples,Constituents found
0,Catastrophizing,A cognitive distortion characterized by predicting the worst possible outcome and treating that outcome as an inevitable or unmanageable certainty.,11.8% (2/17),33.3% (6/18),16.1% (9/56),0.93 (0.11),Catastrophizing: A typo in my report means my manager will fire me and my career is over.my manager will fire me and my career is overA typo in my report means my manager will fire me and my career is over.my manager will fire meMissing one bus means the whole day is ruined and I will never succeed.,catastrophizingfortune telling
1,Cognitive Distortion,"Biased or irrational thought patterns, such as overgeneralization and all-or-nothing thinking, that misinterpret reality and reinforce negative emotional states.",17.6% (3/17),50.0% (9/18),28.6% (16/56),0.90 (0.09),A typo in my report means my manager will fire me and my career is over.my career is overthe whole day is ruinedI will never succeedMissing one bus means... I will never succeed.'I always fail'I always fail,cognitive distortionall or nothing thinkingovergeneralization
2,Cognitive Restructuring,"A therapeutic technique involving the identification, evaluation, and modification of maladaptive thoughts to foster more balanced and evidence-based perspectives.",23.5% (4/17),33.3% (6/18),25.0% (14/56),0.89 (0.09),"Cognitive reappraisal: I list evidence for and against my worry, then consider a balanced explanation.I list evidence for and against my worryconsider a balanced explanationI question the thought 'I always fail' and replace it with a more balanced thought.replace it with a more balanced thoughtCognitive restructuring: I question the thought 'I always fail' and replace it with a more balanced thought.",cognitive restructuringcognitive reappraisalevidence testingbalanced thinking
3,Experiential Avoidance,"The process of attempting to escape or suppress unpleasant internal experiences, such as anxious thoughts or distressing emotions, through behavioral or psychological withdrawal.",23.5% (4/17),33.3% (6/18),23.2% (13/56),0.83 (0.17),I cancel plans whenever I feel anxious so I do not have to face discomfort.I postpone opening the bill because I do not want to feel worried.I do not want to feel worriedAvoidance behavior: I postpone opening the billI do not want to feel worried.,experiential avoidanceavoidance behaviornegative reinforcementemotional regulation


Final tables saved to: /Users/freymon.perez/Projects/GitHub/llm_tracker/discovery_outputs/final_tables


## 10. Build, review, and save a codebook

This step creates your codebook without sending requests to the LLM. Each final
group becomes an entry with a definition and a selection of supporting quotes.
Repeated quote text is listed only once before examples are selected. Examples
are drawn from the group's full set of quotes; there is no guarantee that every
discovered construct in the group will have an example.

Review the definitions and examples before using the codebook. When editing
examples in `codebook_frame`, keep them as lists, such as `["First quote",
"Second quote"]`, rather than one piece of text separated by semicolons.
A quote may itself contain a semicolon, so using semicolons as separators could
split it incorrectly.

The settings in `build_codebook()` mean:

| Setting | What it means |
|---|---|
| `merged` | The final groups you decided to keep, here `final_merged`, including any revised names and definitions. This uses the group information, not the on-screen summary table. |
| `name` | A descriptive title stored with the codebook, such as `"Interview wellbeing codebook"`. It helps identify the codebook; it does not set the saved filename. |
| `version` | A whole-number revision label that you assign, stored with the codebook's descriptive information (metadata). Use `1` for your initial codebook and, for example, `2` for a later reviewed revision. See the guidance below. |
| `n_examples` | The maximum number of supporting quotes per construct, taken from `n_codebook_examples` set earlier. Fewer are included if fewer distinct quotes are available. This does not remove quotes from the discovery results or Step 9 tables. |
| `seed` | A number controlling the random selection of examples when there are more quotes than needed. Keeping it unchanged with the same input selects the same examples. It does not assess their quality or guarantee an example from every constituent. |
| `output_path` | An optional destination ending in a JSON filename for saving immediately. It is omitted here so you can review and edit first, then save in the following cell. Saving adds a date and time to the filename and reports the actual saved path. |

**How to use `version`:** Think of it as the edition of your research coding
manual. You might retain `version=1` while reviewing your first draft, then set
`version=2` when you release revised definitions or groupings to your coding team.
Keep a separate note of what changed so the label is meaningful to collaborators.
The notebook does **not** increase this number automatically, compare editions,
or retrieve an earlier codebook when you change it. Changing only `version` does
not change definitions, examples, or coding decisions. It is unrelated to the
LLM model version, the software version, or the timestamp added to saved files.

This cell creates the codebook (`codebook`) using existing results; it does not
generate new definitions or send paid model requests. The following cell saves
the reviewed table back into the codebook format, retaining its metadata.

Each entry also has space for counterexamples: texts that should **not** be coded
as that construct. These start empty; you can add them during your review.


In [22]:
codebook = build_codebook(
    merged=final_merged,
    name="Discovery tutorial codebook",
    version=1,
    n_examples=n_codebook_examples,
    seed=seed,
)
codebook_frame = codebook_to_frame(codebook)
# Display a readable copy; codebook_frame keeps list values for editing.
display_multiline_table(codebook_frame, ["examples", "counter_examples"])


,construct,definition,examples,counter_examples
0,Catastrophizing,A cognitive distortion characterized by predicting the worst possible outcome and treating that outcome as an inevitable or unmanageable certainty.,my manager will fire me and my career is overmy manager will fire meCatastrophizing: A typo in my report means my manager will fire me and my career is over.,
1,Cognitive Distortion,"Biased or irrational thought patterns, such as overgeneralization and all-or-nothing thinking, that misinterpret reality and reinforce negative emotional states.",I always failthe whole day is ruinedmy career is over,
2,Cognitive Restructuring,"A therapeutic technique involving the identification, evaluation, and modification of maladaptive thoughts to foster more balanced and evidence-based perspectives.","I list evidence for and against my worryCognitive reappraisal: I list evidence for and against my worry, then consider a balanced explanation.replace it with a more balanced thought",
3,Experiential Avoidance,"The process of attempting to escape or suppress unpleasant internal experiences, such as anxious thoughts or distressing emotions, through behavioral or psychological withdrawal.",Avoidance behavior: I postpone opening the billI postpone opening the bill because I do not want to feel worried.I do not want to feel worried.,


In [23]:
# Make any manual edits to codebook_frame before running this cell.
# Example:
# codebook_frame.at[0, "definition"] = "A clearer definition."
# codebook_frame.at[0, "counter_examples"] = ["A quote that should not count."]

if codebook_frame["definition"].fillna("").str.strip().eq("").any():
    raise ValueError("Fill in missing codebook definitions before saving.")

reviewed_codebook = frame_to_codebook(
    codebook_frame,
    metadata=codebook["metadata"],
)
saved_codebook_path = save_codebook(
    reviewed_codebook,
    codebook_dir / "discovered_codebook.json",
)
print("Use this codebook path in the coding pipeline:", saved_codebook_path.resolve())


Saved codebook to /Users/freymon.perez/Projects/GitHub/llm_tracker/discovery_outputs/codebooks/discovered_codebook_20261117_165155_992458.json
Use this codebook path in the coding pipeline: /Users/freymon.perez/Projects/GitHub/llm_tracker/discovery_outputs/codebooks/discovered_codebook_20261117_165155_992458.json


## Next: use your codebook in the main tutorial

Continue with the [main tutorial in Google Colab](https://colab.research.google.com/github/childmindresearch/llm_tracker/blob/main/tutorials/tutorial.ipynb)
to use your saved codebook to find constructs in new documents and compare the
results with human coding.
